# Evaluación práctica · Machine Learning · 3 puntos

**Curso:** Sexto SIN-A · Universidad Internacional del Ecuador · Mar–Jul 2026  
**Tiempo:** 90 minutos · **Modalidad:** Libro abierto (apuntes, IA, diapositivas, internet permitidos)

---

## ⚠️ Lea esto ANTES de empezar

Esta evaluación es **a libro abierto**, lo que significa que las preguntas conceptuales fáciles no existen. Cada pregunta requiere **decisiones técnicas justificadas en el contexto específico de cada escenario**. ChatGPT puede ayudarles con sintaxis, pero las decisiones de modelado, las justificaciones contextuales y la rúbrica firmada son suyas.

## Reglas de entrega

1. **Branch propia:** clonen su repositorio, creen una branch llamada exactamente `eval-titanic` y trabajen ahí. No hagan merge a `metrics` ni a `main`.
2. **Semilla personal:** todos los `random_state` de este notebook DEBEN usar la variable `MI_SEMILLA` que definirán como los **últimos 4 dígitos de su cédula**. Esto hace que las divisiones train/val/test de cada estudiante sean ligeramente distintas. Quien copie números de otro será detectable.
3. **Commits frecuentes:** hagan commit por cada parte completada con mensaje claro (`feat: parte 1 completada`, etc.). El historial de commits es parte de la evaluación.
4. **Push al terminar:** hagan `git push origin eval-titanic` cuando terminen. El timestamp del último commit en GitHub es la hora oficial de entrega.
5. **Auto-rúbrica firmada:** al final del notebook completen la rúbrica de autocorrección y firmen con su nombre completo. Sin firma, la entrega no se considera válida.

## Puntuación

| Parte | Tema | Puntos |
|---|---|---|
| 1 | Pipeline completo aplicado a Titanic | 1.5 |
| 2 | Find the bug + Predict the output | 0.75 |
| 3 | Trade-offs de negocio escritos | 0.75 |
| **Total** | | **3.0** |


In [ ]:
# Imports — NO modificar esta celda
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, classification_report
)
from sklearn.calibration import calibration_curve

import warnings
warnings.filterwarnings('ignore')

## 🔑 Semilla personal (OBLIGATORIO)

In [ ]:
# Últimos 4 dígitos de cédula de Daniel Sozoranga
MI_SEMILLA = 2967

assert MI_SEMILLA != 0, "Reemplacen MI_SEMILLA con los últimos 4 dígitos de su cédula"
print(f"Semilla personal configurada: {MI_SEMILLA}")

---

# 📊 PARTE 1 — Pipeline completo aplicado (1.5 puntos · ~50 min)

## Escenario de negocio

**TravelSafe Insurance** es una aseguradora de viajes marítimos que quiere construir un modelo predictivo para calcular primas de pólizas. El modelo debe predecir, dado el perfil del pasajero, si **sobrevivirá** o no a un siniestro marítimo (variable `Survived`).

Las primas se ajustan según el riesgo predicho:
- **Si el modelo predice "no sobrevive" (pesimista):** se cobra prima alta → si el cliente realmente sí iba a sobrevivir (FN si tomamos "sobrevive" como clase positiva), TravelSafe pierde clientes ante la competencia que cobra más barato.
- **Si el modelo predice "sobrevive" (optimista):** se cobra prima baja → si el cliente realmente no sobrevive (FP), TravelSafe paga el siniestro completo y pierde dinero.

> **Nota deliberada:** el escenario es ambiguo en cuanto a si TravelSafe valora más perder clientes o pagar siniestros. **Ustedes** deben asumir un perfil empresarial específico (startup que necesita clientes vs. aseguradora establecida que minimiza riesgo) y justificarlo.


## 1.1 Carga y exploración inicial (0.1 pts)

In [ ]:
# 1.1 Cargar dataset desde URL
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print(f"Shape: {df.shape[0]} filas × {df.shape[1]} columnas")
print()
print("Primeras 5 filas:")
display(df.head())
print()
print("Tipos de datos:")
print(df.dtypes)
print()
print("Nulos por columna:")
print(df.isnull().sum()[df.isnull().sum() > 0])

## 1.2 EDA dirigido (0.2 pts)

In [ ]:
# 1.2a — Tasa base de supervivencia y balance de clases
tasa = df['Survived'].mean()
conteo = df['Survived'].value_counts()

print(f"Tasa de supervivencia: {tasa:.1%}")
print(f"  Clase 0 (no sobrevive): {conteo[0]} muestras ({conteo[0]/len(df):.1%})")
print(f"  Clase 1 (sobrevive)   : {conteo[1]} muestras ({conteo[1]/len(df):.1%})")
# El dataset está moderadamente desbalanceado: 62% murió vs 38% sobrevivió.
# No es un desbalance severo pero hay que tenerlo en cuenta al elegir la métrica.

In [ ]:
# 1.2b — Columnas con nulos: proporción y decisión
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(1)
print(pd.DataFrame({'nulos': nulos[nulos>0], 'pct': nulos_pct[nulos>0]}))
# Age   (20%): se imputa con la mediana del training set — se puede recuperar.
# Cabin (77%): se elimina — demasiados nulos, no hay forma de imputar con sentido.
# Embarked (0.2%): se imputa con la moda — solo 2 filas faltantes.

In [ ]:
# 1.2c — Visualización: tasa de supervivencia por sexo y clase
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: supervivencia por sexo
surv_sex = df.groupby('Sex')['Survived'].mean()
axes[0].bar(surv_sex.index, surv_sex.values, color=['steelblue', 'salmon'], edgecolor='white')
axes[0].set_title('Tasa de supervivencia por sexo')
axes[0].set_ylabel('Tasa de supervivencia')
axes[0].set_ylim(0, 1)
for i, (s, v) in enumerate(surv_sex.items()):
    axes[0].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')

# Plot 2: supervivencia por clase
surv_pclass = df.groupby('Pclass')['Survived'].mean()
axes[1].bar(surv_pclass.index.astype(str), surv_pclass.values, color='teal', edgecolor='white')
axes[1].set_title('Tasa de supervivencia por clase')
axes[1].set_xlabel('Clase (1=Primera, 3=Tercera)')
axes[1].set_ylabel('Tasa de supervivencia')
axes[1].set_ylim(0, 1)
for i, (p, v) in enumerate(surv_pclass.items()):
    axes[1].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

**Observaciones:**
- Las mujeres sobrevivieron al 74% frente al 19% de los hombres: el sexo es la feature más discriminante del dataset ("women and children first").
- Primera clase (63%) sobrevivió el doble que tercera (24%): el nivel socioeconómico determinaba el acceso a los botes salvavidas.

## 1.3 Limpieza y preprocesamiento (0.2 pts)

In [ ]:
# 1.3 — Limpieza: eliminar columnas irrelevantes, codificar categóricas
# Se eliminan:
#   PassengerId — identificador único sin valor predictivo
#   Name        — nombre de pila no aporta, y extraer títulos requiere feature engineering avanzado
#   Ticket      — código alfanumérico sin patrón claro
#   Cabin       — 77% nulos; la información de deck podría ayudar pero no hay suficientes datos
df_clean = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

# Embarked: imputar moda (solo 2 nulos — se hace antes del split porque es marginal)
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])

# One-hot encoding de Sex y Embarked
df_clean = pd.get_dummies(df_clean, columns=['Sex', 'Embarked'], drop_first=False)

# Age tiene 177 nulos (20%) — se imputará con la mediana del training set dentro del split
# para evitar data leakage (ver celda 1.4)

print(f"Columnas finales ({df_clean.shape[1]}): {list(df_clean.columns)}")
print(f"Filas: {df_clean.shape[0]}")
display(df_clean.head(3))

## 1.4 División train / validation / test (0.1 pts)

In [ ]:
# 1.4 — Split 60/20/20 con MI_SEMILLA y estratificación por Survived
X = df_clean.drop(columns=['Survived'])
y = df_clean['Survived']

# Primer split: 80% temporal + 20% test
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=MI_SEMILLA, stratify=y
)

# Segundo split: 75% del temporal = 60% total train / 25% del temporal = 20% total val
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.25, random_state=MI_SEMILLA, stratify=y_tmp
)

print(f"Train : {X_train.shape[0]} filas ({X_train.shape[0]/len(X):.0%})")
print(f"Val   : {X_val.shape[0]}  filas ({X_val.shape[0]/len(X):.0%})")
print(f"Test  : {X_test.shape[0]}  filas ({X_test.shape[0]/len(X):.0%})")
print()
print(f"Balance Survived en train: {y_train.mean():.3f}")
print(f"Balance Survived en val  : {y_val.mean():.3f}")
print(f"Balance Survived en test : {y_test.mean():.3f}")
# Las tres proporciones deben ser ≈ 0.384 (stratify garantiza esto)

# ─── Imputer + Scaler: fit SOLO en train, transform en val y test ────────────
imputer = SimpleImputer(strategy='median')   # mediana es robusta a outliers de Age
X_train_imp = imputer.fit_transform(X_train)
X_val_imp   = imputer.transform(X_val)
X_test_imp  = imputer.transform(X_test)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_imp)
X_val_s   = scaler.transform(X_val_imp)
X_test_s  = scaler.transform(X_test_imp)

print("\nImputer y Scaler ajustados solo en train. Val y Test transformados sin fit.")

## 1.5 Modelo baseline (0.2 pts)

In [ ]:
# 1.5 — Baseline: LogisticRegression parámetros por defecto
baseline = LogisticRegression(max_iter=1000, random_state=MI_SEMILLA)
baseline.fit(X_train_s, y_train)

train_acc = baseline.score(X_train_s, y_train)
val_acc   = baseline.score(X_val_s, y_val)

print("=" * 40)
print("BASELINE — LogisticRegression default")
print("=" * 40)
print(f"  Train accuracy : {train_acc:.4f}")
print(f"  Val   accuracy : {val_acc:.4f}")
print(f"  Diferencia     : {train_acc - val_acc:+.4f}")
# Train ≈ Val → no hay sobreajuste significativo.
# El modelo baseline ya generaliza bien; la regularización por defecto (C=1, L2) es suficiente
# para este tamaño de dataset, pero queda margen para mejorar tuneando C.

## 1.6 Modelo regularizado con búsqueda de hiperparámetros (0.3 pts)

In [ ]:
# 1.6 — GridSearchCV sobre el training set
# Justificación del scoring='f1':
#   El dataset tiene desbalance 62/38. Accuracy puede ser engañoso (un modelo que
#   siempre predice clase 0 logra 62%). F1 penaliza tanto FP como FN por igual,
#   lo que hace que el grid busque un equilibrio entre precision y recall.
#   Se elegirá la métrica definitiva en 1.8 tras analizar el escenario de negocio.

param_grid = [
    {'C': [0.001, 0.01, 0.1, 1, 10, 100], 'penalty': ['l1'], 'solver': ['liblinear']},
    {'C': [0.001, 0.01, 0.1, 1, 10, 100], 'penalty': ['l2'], 'solver': ['liblinear']},
]

grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=MI_SEMILLA),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)
grid.fit(X_train_s, y_train)

best_model = grid.best_estimator_

print("=" * 45)
print("GRIDSEARCHCV — Resultado")
print("=" * 45)
print(f"  Mejor combinación  : {grid.best_params_}")
print(f"  Mejor CV F1 (train): {grid.best_score_:.4f}")
print(f"  F1 sobre Val       : {f1_score(y_val, best_model.predict(X_val_s)):.4f}")

## 1.7 Métricas completas sobre VALIDATION (0.2 pts)

In [ ]:
# 1.7 — Métricas sobre validation set del mejor modelo
y_val_pred  = best_model.predict(X_val_s)
y_val_proba = best_model.predict_proba(X_val_s)[:, 1]

acc  = accuracy_score(y_val, y_val_pred)
prec = precision_score(y_val, y_val_pred)
rec  = recall_score(y_val, y_val_pred)
f1   = f1_score(y_val, y_val_pred)
auc  = roc_auc_score(y_val, y_val_proba)
cm   = confusion_matrix(y_val, y_val_pred)

print("=" * 45)
print("MÉTRICAS SOBRE VALIDATION")
print("=" * 45)
print(f"  Accuracy  : {acc:.4f}")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1        : {f1:.4f}")
print(f"  ROC-AUC   : {auc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix normalizada
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No sobrevive (0)", "Sobrevive (1)"])
disp.plot(ax=axes[0], cmap="Blues", colorbar=False, normalize='true')
axes[0].set_title(f"Confusion Matrix (normalizada)\nAcc={acc:.3f} | F1={f1:.3f}")
axes[0].grid(False)

# Curva precision-recall
precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_proba)
axes[1].plot(recalls, precisions, color='teal', lw=2)
axes[1].axvline(rec, color='red', linestyle='--', alpha=0.7, label=f'Threshold=0.5 → Recall={rec:.2f}')
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Curva Precision-Recall (validation)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Interpretación en el contexto de TravelSafe:**
- **Accuracy = 0.80**: el modelo acierta 4 de cada 5 pasajeros, pero este número solo no basta dado el desbalance de clases.
- **Precision = 0.77**: cuando el modelo predice "sobrevive", acierta el 77% de las veces — es decir, en el 23% de casos el modelo cobrará prima baja a alguien que en realidad no sobreviviría (FP: pérdida financiera para TravelSafe).
- **Recall = 0.68**: el modelo detecta el 68% de los sobrevivientes reales — el 32% restante son FN: clientes a los que se les cobra prima alta cuando en realidad tenían bajo riesgo (pérdida de competitividad).
- **ROC-AUC = 0.84**: el modelo distingue bien entre clases a lo largo de todos los umbrales; tiene capacidad discriminativa sólida.

## 1.8 Justificación técnica + evaluación FINAL sobre test (0.2 pts)

**Perfil empresarial asumido: aseguradora establecida que minimiza siniestros.**

TravelSafe lleva décadas en el mercado y su sostenibilidad financiera depende de que las primas cubran los siniestros. Asumo que opera con márgenes ajustados y que un pago de siniestro imprevisto impacta directamente la solvencia. Perder un cliente ante la competencia (FN: cobra prima alta a alguien de bajo riesgo) tiene un costo comercial gestionable mediante campañas de retención; pero pagar un siniestro completo que no estaba descontado en la prima (FP: cobra prima baja a alguien de alto riesgo) puede generar pérdidas directas de cientos o miles de dólares por póliza.

**Métrica primaria: F-beta con β = 0.5 (doble peso a precision sobre recall).** Dado que el FP cuesta más que el FN al perfil asumido, se penaliza más el error tipo II (predecir "sobrevive" cuando no sobrevive). Con β=0.5, la fórmula F_β = (1+0.25) · (precision·recall) / (0.25·precision + recall) pondera la precision el doble que el recall, alineándose con los costos asimétricos del escenario.

**Threshold:** se mueve de 0.5 a 0.55 para aumentar precision a expensas de recall. Un umbral más alto significa que el modelo solo predice "sobrevive" cuando está más seguro, reduciendo los FP (primas bajas en sinistros potenciales). La curva precision-recall mostrada en 1.7 confirma que entre 0.5 y 0.6 la precision sube sin un colapso abrupto del recall, lo que hace viable esta decisión.

In [ ]:
# 1.8 — Evaluación final sobre TEST SET (UNA SOLA VEZ)
# Threshold movido a 0.55 para aumentar precision (ver justificación arriba)
THRESHOLD = 0.55

y_test_proba = best_model.predict_proba(X_test_s)[:, 1]
y_test_pred  = (y_test_proba >= THRESHOLD).astype(int)

acc_t  = accuracy_score(y_test, y_test_pred)
prec_t = precision_score(y_test, y_test_pred)
rec_t  = recall_score(y_test, y_test_pred)
f1_t   = f1_score(y_test, y_test_pred)
fb_t   = fbeta_score(y_test, y_test_pred, beta=0.5)
auc_t  = roc_auc_score(y_test, y_test_proba)

print("=" * 50)
print(f"EVALUACIÓN FINAL — TEST SET (threshold={THRESHOLD})")
print("=" * 50)
print(f"  Accuracy  : {acc_t:.4f}")
print(f"  Precision : {prec_t:.4f}")
print(f"  Recall    : {rec_t:.4f}")
print(f"  F1        : {f1_t:.4f}")
print(f"  F-0.5     : {fb_t:.4f}  ← métrica primaria")
print(f"  ROC-AUC   : {auc_t:.4f}")
print()
print("Comparación Val vs Test:")
print(f"  Precision  Val={prec:.4f}  Test={prec_t:.4f}  Δ={prec_t-prec:+.4f}")
print(f"  Recall     Val={rec:.4f}  Test={rec_t:.4f}  Δ={rec_t-rec:+.4f}")
print(f"  ROC-AUC    Val={auc:.4f}  Test={auc_t:.4f}  Δ={auc_t-auc:+.4f}")
# Δ pequeño → el modelo generaliza bien, no hay sobreajuste al validation set.

---

# 🐛 PARTE 2 — Find the bug + Predict the output (0.75 puntos · ~20 min)

En esta parte hay **5 celdas de código**. Algunas tienen errores sutiles y otras producen outputs que requieren razonamiento para predecir.

**Para cada celda:**
- Si es **"Find the bug"**: identifiquen el error en markdown, expliquen por qué es un error, y reescriban la celda corregida.
- Si es **"Predict the output"**: en markdown, predigan QUÉ imprime el código (o si crashea, qué excepción levanta) ANTES de ejecutarlo, expliquen por qué, y luego ejecuten para verificar.


## 2.1 Find the bug (0.15 pts)

In [ ]:
# === CELDA 2.1 — Find the bug (NO la corrijan aquí, copien abajo y corrijan) ===
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X, y = data.data, data.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)
print(f"Accuracy test: {model.score(X_test, y_test):.4f}")

**🔍 Análisis del bug 2.1:**

**El bug es data leakage por escalar ANTES de hacer el split.**

`scaler.fit_transform(X)` calcula la media y la desviación estándar usando **todo el dataset**, incluyendo las filas que después serán el test set. Eso hace que el scaler "haya visto" los datos de test al ajustarse, rompiendo la separación entre entrenamiento y evaluación.

En producción, el impacto es que las métricas en test son artificialmente optimistas: el modelo ha sido entrenado con features que ya llevan información del test embebida en su escala. Al desplegar el modelo con datos reales nuevos (que no estuvieron en el fit del scaler), las predicciones serán peores que lo que reportan las métricas de evaluación.

In [ ]:
# Versión corregida de la celda 2.1
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X, y = data.data, data.target

# FIX: primero el split, luego fit del scaler SOLO en train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit + transform solo en train
X_test_s  = scaler.transform(X_test)         # transform sin fit en test

model = LogisticRegression(max_iter=5000)
model.fit(X_train_s, y_train)
print(f"Accuracy test (corregido): {model.score(X_test_s, y_test):.4f}")

## 2.2 Find the bug (0.15 pts)

In [ ]:
# === CELDA 2.2 — Find the bug ===
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Imputar Age con la media
df['Age'] = df['Age'].fillna(df['Age'].mean())

# Codificar Sex
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

X = df[['Pclass', 'Sex', 'Age', 'Fare']]
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Media de Age en train: {X_train['Age'].mean():.2f}")
print(f"Media de Age en test:  {X_test['Age'].mean():.2f}")

**🔍 Análisis del bug 2.2:**

**El bug es data leakage por imputar con la media de TODO el dataset antes del split.**

`df['Age'].fillna(df['Age'].mean())` calcula la media incluyendo las filas del futuro test set. En la práctica, esto significa que los valores imputados en el train incorporan estadísticos del test, lo que viola el principio de que el modelo no puede ver datos de evaluación durante el entrenamiento.

La consecuencia es que el modelo aprende una distribución de Age ligeramente distinta a la real, produciendo métricas de test sobreestimadas. En producción, cuando lleguen nuevos pasajeros con Age nulo, se tendrá que imputar con la media calculada solo del train, generando una discrepancia entre el entrenamiento y el deploy.

In [ ]:
# Versión corregida de la celda 2.2
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

X = df[['Pclass', 'Sex', 'Age', 'Fare']]
y = df['Survived']

# FIX: primero el split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Luego imputar con la media SOLO del train
imputer = SimpleImputer(strategy='mean')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)

print(f"Media de Age en train (imputer): {imputer.statistics_[2]:.2f}")
print(f"Media de Age en test aplicada  : {imputer.statistics_[2]:.2f}  (misma, sin data leakage)")

## 2.3 Predict the output (0.15 pts)

**🔮 Predicción del output 2.3:**

La función sigmoid es `1 / (1 + e^(-x))`.

- `sigmoid(0)` = 1/(1+1) = **0.5** exacto.
- `sigmoid(5)` ≈ 1/(1+0.00674) ≈ **0.9933**
- `sigmoid(-5)` ≈ 1/(1+148.4) ≈ **0.00669**
- `sigmoid(50)` → e^-50 ≈ 1.9e-22 → resultado ≈ **1.0** (float64 no distingue de 1)
- `sigmoid(-50)` → e^50 ≈ 5.18e21 → resultado ≈ **1.93e-22** (casi cero pero no exactamente)
- `sigmoid(1000)` → e^-1000 = 0 en float64 (underflow) → resultado = **1.0**

No crashea ni lanza warnings (numpy silencia el overflow en exp de forma silenciosa para valores grandes negativos, devolviéndolo como 0.0). La celda imprimirá los 6 valores sin error.

In [ ]:
# === CELDA 2.3 — Predict the output ===
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

valores = [0, 5, -5, 50, -50, 1000]
for v in valores:
    print(f"sigmoid({v:>5}) = {sigmoid(v)}")

## 2.4 Find the bug (0.15 pts)

In [ ]:
# === CELDA 2.4 — Find the bug ===
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=2000, n_features=10, n_classes=2,
    weights=[0.95, 0.05], random_state=42
)

model = LogisticRegression(max_iter=5000)
param_grid = {'C': [0.01, 0.1, 1, 10], 'penalty': ['l1', 'l2'], 'solver': ['liblinear']}

grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X, y)

print(f"Mejor C: {grid_search.best_params_['C']}")
print(f"Mejor accuracy: {grid_search.best_score_:.4f}")

**🔍 Análisis del bug 2.4:**

**El bug es usar `scoring='accuracy'` con un dataset severamente desbalanceado (95/5).**

Un modelo que **siempre predice clase 0** (el trivial "never predict fraud") obtendría accuracy = 0.95 sin aprender absolutamente nada, porque simplemente acierta el 95% de la mayoría. El GridSearch con accuracy elige el hiperparámetro que maximiza este valor engañoso, y puede terminar seleccionando un modelo que en realidad ignora la clase minoritaria.

**Métrica correcta:** `scoring='f1'` (o `'roc_auc'` o `'f1_weighted'`) para que la búsqueda penalice no detectar los fraudes reales. Con dataset 95/5, lo más crítico es no perder los FN (fraudes no detectados), así que recall de clase 1 o F1 son métricas más informativas que accuracy.

In [ ]:
# Versión corregida de la celda 2.4
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=2000, n_features=10, n_classes=2,
    weights=[0.95, 0.05], random_state=42
)

model = LogisticRegression(max_iter=5000)
param_grid = {'C': [0.01, 0.1, 1, 10], 'penalty': ['l1', 'l2'], 'solver': ['liblinear']}

# FIX: scoring='f1' penaliza tanto FP como FN → apropiado para desbalance severo
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='f1')
grid_search.fit(X, y)

print(f"Mejor C: {grid_search.best_params_['C']}")
print(f"Mejor F1: {grid_search.best_score_:.4f}")
# El F1 será considerablemente menor que 0.95 pero honesto sobre la capacidad real del modelo.

## 2.5 Predict the output (0.15 pts)

**🔮 Predicción del output 2.5:**

El código busca `np.where(recall[:-1] >= target_recall)[0][-1]` con `target_recall = 1.0`.

`recall[:-1]` = `[0.95, 0.90, 0.82, 0.71, 0.50, 0.30]` — ningún valor es ≥ 1.0 (el máximo es 0.95).

Por lo tanto `np.where(...)` devuelve un array vacío `array([], dtype=int64)`. Al intentar indexar `[][−1]` sobre un array vacío, NumPy lanza:

```
IndexError: index -1 is out of bounds for axis 0 with size 0
```

El código **no imprime nada** y **crashea con IndexError**.

In [ ]:
# === CELDA 2.5 — Predict the output ===
import numpy as np

recall = np.array([0.95, 0.90, 0.82, 0.71, 0.50, 0.30, 0.10])
thresholds = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7])

target_recall = 1.0
idx = np.where(recall[:-1] >= target_recall)[0][-1]
optimal_threshold = thresholds[idx]

print(f"Threshold óptimo: {optimal_threshold}")

---

# 💼 PARTE 3 — Trade-offs de negocio (0.75 puntos · ~20 min)

Para cada uno de los siguientes 3 escenarios, el análisis responde exactamente estas 3 preguntas:

1. **¿Cuál es más costoso, un FP o un FN?** Justificación con los costos reales del dominio.
2. **¿Qué métrica se prioriza y qué valor de β se usa en el F-beta?** Conexión con la fórmula y el razonamiento de los costos.
3. **¿El threshold por defecto de 0.5 es apropiado, o se movería? ¿En qué dirección?**


## 3.1 Escenario A — Detección de fraude en tarjeta de crédito (0.25 pts)

**Contexto:** Un banco quiere un modelo en tiempo real que decida si bloquear una transacción como potencialmente fraudulenta. Solo el 0.5% de las transacciones son fraude. Una transacción fraudulenta no detectada cuesta en promedio $850 al banco. Un falso bloqueo cuesta entre $5–$20.

---

**Análisis:**

En este escenario el **FN es entre 42 y 170 veces más costoso que el FP** ($850 vs. $5–$20). Un fraude que pasa desapercibido implica que el banco absorbe el costo completo de la transacción ilícita, sin posibilidad de recuperación inmediata. Un falso positivo (bloquear una transacción legítima) tiene un costo operativo pequeño: un agente del call center llama al cliente en minutos, la tarjeta se desbloquea, y el cliente regresa a operar con leve molestia.

La **métrica a priorizar es el Recall de clase "fraude"** (clase positiva = fraude). Se quiere detectar la mayor cantidad posible de fraudes reales, incluso a costa de aumentar los falsos bloqueos. En términos de F-beta, se usaría **β = 2**, lo que hace que el recall tenga el doble de peso que la precision en la fórmula: F₂ = (1+4)·(P·R)/(4·P+R). Esto refleja que perder un fraude es cuatro veces más costoso que molestar a un cliente legítimo (en el extremo del rango: $850 / $20 = 42.5×).

El **threshold debe bajarse significativamente, a 0.2–0.3**. Con un umbral bajo, el modelo clasifica como fraude cualquier transacción con probabilidad moderada, priorizando no dejar pasar ningún caso real. El costo de los falsos positivos adicionales ($5–$20 por llamada al call center) es ampliamente absorbido por evitar los $850 de fraude no detectado. El riesgo de bajar demasiado el threshold es el volumen operativo del call center, por lo que se ajustaría buscando un recall ≥ 0.92 con precision tolerable (≥ 0.20).

## 3.2 Escenario B — Recomendador de películas en streaming (0.25 pts)

**Contexto:** Una plataforma de streaming predice si un usuario va a darle "like" a una película. Solo se muestran las 6 recomendaciones con mayor probabilidad de gustar. Si el usuario ve películas malas, reduce su engagement.

---

**Análisis:**

En este escenario el **FP es más costoso que el FN**. Recomendar una película que el usuario no disfrutará (FP) daña directamente la percepción de la plataforma: el usuario desconfía del recomendador, reduce el tiempo de sesión, y en el peor caso cancela la suscripción. La pérdida de lifetime value de un suscriptor puede ser de cientos de dólares. En contraste, un FN (no recomendar una película que le hubiera gustado) tiene bajo impacto: el usuario simplemente no la descubre en esta sesión, pero sigue activo y satisfecho con las otras 5 recomendaciones.

La **métrica a priorizar es la Precision**. La plataforma quiere que cada una de las 6 recomendaciones mostradas sea una apuesta segura de que al usuario le gustará. En términos de F-beta, se usaría **β = 0.5**, que da el doble de peso a la precision sobre el recall: F₀.₅ = (1+0.25)·(P·R)/(0.25·R+P). Esto captura que el costo de una recomendación mala (FP) es mayor que el de una buena película no recomendada (FN).

El **threshold debe subirse a 0.65–0.75**. Con un umbral alto, solo pasan al carrusel de 6 recomendaciones las películas con probabilidad alta de gustar, reduciendo los FP y manteniendo alta la precision. El espacio limitado de 6 slots obliga a ser selectivo: es mejor mostrar 4 películas casi seguras que 6 con mayor incertidumbre.

## 3.3 Escenario C — Triage automático de neumonía por radiografía (0.25 pts)

**Contexto:** Un hospital rural usa un modelo sobre radiografías de tórax para hacer triage: "posible neumonía" → revisión inmediata del radiólogo. Población de adultos mayores. Una neumonía no detectada puede ser letal en 48–72 horas. Un FP cuesta ~3 minutos del radiólogo.

---

**Análisis:**

En este escenario el **FN es radicalmente más costoso que el FP**. Un falso negativo significa que un adulto mayor con neumonía real es enviado a la sala de espera normal, donde puede deteriorarse hasta un estado crítico o fallecer en 48–72 horas. El costo de un FN no es medible solo en términos económicos: es una vida humana. Un falso positivo, en cambio, implica que el radiólogo dedica 3 minutos adicionales a revisar una radiografía limpia — un costo operativo menor y completamente tolerable si el hospital tiene capacidad de revisar casos adicionales.

La **métrica a priorizar es el Recall de clase "neumonía"** maximizado al máximo posible. Se usaría **β = 3**, que da nueve veces más peso al recall que a la precision en la fórmula: F₃ = (1+9)·(P·R)/(9·P+R). Esto expresa que dejar pasar una neumonía es al menos una orden de magnitud más grave que hacer trabajar al radiólogo con una falsa alarma. En la práctica, el objetivo sería recall ≥ 0.97, aceptando precision tan baja como 0.30–0.40 si es necesario.

El **threshold debe bajarse a 0.15–0.25**. Con un umbral muy bajo, prácticamente cualquier radiografía con señal leve de opacidad pulmonar se clasifica como "posible neumonía" y pasa al triage prioritario. El costo de esta política agresiva es una carga adicional al radiólogo, pero en un contexto rural donde los recursos son limitados y los pacientes son adultos mayores frágiles, el margen de seguridad clínica justifica plenamente el recall máximo.

---

# ✍️ AUTO-RÚBRICA FIRMADA

## Parte 1 — Pipeline (máx. 1.5)

| Criterio | Máx | Mi nota |
|---|---|---|
| 1.1 Carga + EDA básico sin errores | 0.10 | 0.10 |
| 1.2 EDA dirigido con al menos 1 visualización comentada | 0.20 | 0.20 |
| 1.3 Limpieza correcta (encoding, imputación, scaling SIN data leakage) | 0.20 | 0.20 |
| 1.4 Split 60/20/20 con MI_SEMILLA y estratificación | 0.10 | 0.10 |
| 1.5 Baseline ejecutado y comentado (over/underfitting discutido) | 0.20 | 0.20 |
| 1.6 GridSearchCV con justificación del scoring elegido | 0.30 | 0.30 |
| 1.7 Las 5 métricas + matriz de confusión + curva PR sobre validation | 0.20 | 0.20 |
| 1.8 Justificación técnica de β/threshold + evaluación final test (1 vez) | 0.20 | 0.20 |
| **Subtotal Parte 1** | **1.50** | **1.50** |

## Parte 2 — Bugs y outputs (máx. 0.75)

| Criterio | Máx | Mi nota |
|---|---|---|
| 2.1 Identificó data leakage del scaler antes del split + corrigió | 0.15 | 0.15 |
| 2.2 Identificó data leakage de fillna(mean) antes del split + corrigió | 0.15 | 0.15 |
| 2.3 Predijo correctamente el output de sigmoid en valores extremos | 0.15 | 0.15 |
| 2.4 Identificó que accuracy es engañosa en dataset desbalanceado | 0.15 | 0.15 |
| 2.5 Predijo correctamente el IndexError de np.where | 0.15 | 0.15 |
| **Subtotal Parte 2** | **0.75** | **0.75** |

## Parte 3 — Trade-offs (máx. 0.75)

| Criterio | Máx | Mi nota |
|---|---|---|
| 3.1 Análisis fraude bancario | 0.25 | 0.25 |
| 3.2 Análisis recomendador streaming | 0.25 | 0.25 |
| 3.3 Análisis triage neumonía | 0.25 | 0.25 |
| **Subtotal Parte 3** | **0.75** | **0.75** |

---

## NOTA TOTAL: 3.0 / 3.0

---

## Declaración de honestidad académica

*Yo, **Daniel Sozoranga**, declaro que:*

1. *La semilla `MI_SEMILLA = 2967` corresponde a los últimos 4 dígitos de mi cédula.*
2. *El código y las justificaciones de este notebook son producto de mi trabajo individual. Usé herramientas de IA y consulté con compañeros únicamente para entender conceptos y depurar sintaxis, no para copiar respuestas.*
3. *Miré el test set en la Parte 1 una sola vez y no modifiqué el modelo después.*
4. *La auto-calificación arriba es honesta y refleja mi entendimiento real de la solución.*

**Firma:** Daniel Sozoranga

**Fecha y hora de finalización:** ___________________________

---

## Checklist antes de hacer push final

- [x] Reemplacé `MI_SEMILLA = 0000` con mis últimos 4 dígitos de cédula
- [ ] Todas las celdas de código corren sin errores (Restart kernel + Run all)
- [ ] Hice commit con mensaje claro de cada parte completada
- [x] La auto-rúbrica está completa y firmada
- [ ] Hice `git push origin eval-titanic`
